# Stage 7 – Scoring the rejected applications

Applies the final model to a 300k sample of rejected applications (memory-safe).

Part of **Explainable AI for Credit Risk Prediction**. See the main README for the full pipeline.

In [17]:
# 06_fairness_rejected_memory_safe.ipynb
# ==================================================
# Fairness Analysis (Memory-Safe Version)
# ==================================================

import pandas as pd
import numpy as np
import joblib
import os
import json
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid")

from scipy.stats import ks_2samp

SAVE_DIR = "../outputs/notebook3_outputs"
REJECTED_PATH = "../data/rejected_2007_to_2018Q4.csv"

FINAL_MODEL_PATH = os.path.join(SAVE_DIR, "final_model_xgb.joblib")
CONFIG_JSON_PATH = os.path.join(SAVE_DIR, "final_model_config.json")
PIPELINE_PATH = os.path.join(SAVE_DIR, "preprocessing_pipeline.joblib")

print("Loading model + config + pipeline...")

xgb_model = joblib.load(FINAL_MODEL_PATH)

with open(CONFIG_JSON_PATH, "r") as f:
    config = json.load(f)

feature_names = config["feature_names"]
chosen_threshold = config["chosen_threshold"]
low_cut = config["risk_band_cutoffs"]["low_cutoff"]
high_cut = config["risk_band_cutoffs"]["high_cutoff"]

bundle = joblib.load(PIPELINE_PATH)
numeric_preprocessor = bundle["numeric_preprocessor"]
ordinal_encoder      = bundle["ordinal_encoder"]
target_encoders      = bundle["target_encoders"]
numeric_cols_final   = bundle["numeric_cols_final"]
small_cat_used       = bundle["small_cat_used"]
large_cat_used       = bundle["large_cat_used"]

print("Loaded.")

# -----------------------------------------------------
# 1. Memory-SAFE loading of rejected file
# -----------------------------------------------------

print("\nLoading only 300,000 rows from rejected file (memory safe)...")

rejected = pd.read_csv(
    REJECTED_PATH,
    nrows=300_000,       # 🔥 SAFE LOAD
    low_memory=True
)

print("Loaded rejected sample:", rejected.shape)

# -----------------------------------------------------
# 2. Add missing raw columns (memory safe)
# -----------------------------------------------------

required_raw_cols = list(set(numeric_cols_final + small_cat_used + large_cat_used))
missing_raw_cols  = [c for c in required_raw_cols if c not in rejected.columns]

print("Missing columns:", missing_raw_cols)

# Fill missing numeric safely
for col in missing_raw_cols:
    if col in numeric_cols_final:
        rejected[col] = np.nan   # SAFE: shape is only 300k rows
    else:
        rejected[col] = "__MISSING__"

# Categorical → str
for c in small_cat_used + large_cat_used:
    if c in rejected.columns:
        rejected[c] = rejected[c].astype(str).fillna("__MISSING__")

# -----------------------------------------------------
# 3. Apply SAME preprocessing from Notebook 3
# -----------------------------------------------------

# Numeric
X_num = numeric_preprocessor.transform(rejected[numeric_cols_final])
X_num = np.asarray(X_num, dtype=float)

# Small categorical
if small_cat_used and ordinal_encoder:
    X_small = ordinal_encoder.transform(rejected[small_cat_used].astype(str))
    small_ord_cols = [c + "_ord" for c in small_cat_used]
else:
    X_small = np.empty((rejected.shape[0], 0))
    small_ord_cols = []

# Large categorical TE
te_arrays = []
te_cols = []

for col in large_cat_used:
    mapping = target_encoders[col]
    fallback = float(np.mean(list(mapping.values()))) if len(mapping) else 0.5
    arr = rejected[col].map(mapping).fillna(fallback).values
    te_arrays.append(arr)
    te_cols.append(col + "_te")

X_te = np.vstack(te_arrays).T if te_arrays else np.empty((rejected.shape[0], 0))

# Combine
X_all = pd.DataFrame(X_num, columns=numeric_cols_final)

if X_small.shape[1] > 0:
    X_all = pd.concat([X_all,
                       pd.DataFrame(X_small, columns=small_ord_cols)], axis=1)

if X_te.shape[1] > 0:
    X_all = pd.concat([X_all,
                       pd.DataFrame(X_te, columns=te_cols)], axis=1)

# Reorder
X_all = X_all[feature_names]
X_matrix = X_all.values.astype(float)

print("\nFinal engineered rejected feature matrix:", X_matrix.shape)

# -----------------------------------------------------
# 4. Predict
# -----------------------------------------------------

probs = xgb_model.predict_proba(X_matrix)[:, 1]
rejected["pred_default_prob"] = probs
rejected["pred_label"] = (probs >= chosen_threshold).astype(int)

# -----------------------------------------------------
# 5. Fairness Stats
# -----------------------------------------------------

def fairness_stats(df, col):
    rows = []
    for g in df[col].unique():
        mask = df[col] == g
        if mask.sum() < 150:  # skip tiny groups
            continue
        rows.append({
            "group": g,
            "count": mask.sum(),
            "avg_prob": df.loc[mask, "pred_default_prob"].mean(),
            "rejection_rate": df.loc[mask, "pred_label"].mean()
        })
    return pd.DataFrame(rows).sort_values("avg_prob", ascending=False)

for col in ["purpose", "home_ownership", "emp_length"]:
    if col in rejected.columns:
        print(f"\n=== Fairness by: {col} ===")
        print(fairness_stats(rejected, col))

# -----------------------------------------------------
# 6. Save
# -----------------------------------------------------

rejected.to_csv(os.path.join(SAVE_DIR, "rejected_scored_300k.csv"), index=False)
print("\nSaved scored rejected sample.")
print("✔ Fairness analysis complete (memory safe).")


Loading model + config + pipeline...
Loaded.

Loading only 300,000 rows from rejected file (memory safe)...
Loaded rejected sample: (300000, 9)
Missing columns: ['mo_sin_old_il_acct', 'avg_cur_bal', 'initial_list_status', 'acc_open_past_24mths', 'max_bal_bc', 'open_act_il', 'sub_grade', 'dti', 'all_util', 'delinq_amnt', 'purpose', 'pub_rec_bankruptcies', 'pct_tl_nvr_dlq', 'open_il_24m', 'acc_now_delinq', 'tax_liens', 'emp_length', 'inq_last_6mths', 'annual_inc', 'num_accts_ever_120_pd', 'emp_title', 'num_op_rev_tl', 'disbursement_method', 'num_rev_accts', 'open_rv_24m', 'grade', 'pub_rec', 'total_rev_hi_lim', 'open_acc', 'revol_util', 'percent_bc_gt_75', 'tot_coll_amt', 'mo_sin_old_rev_tl_op', 'total_acc', 'num_tl_30dpd', 'open_acc_6m', 'mo_sin_rcnt_rev_tl_op', 'inq_fi', 'funded_amnt_inv', 'fico_range_low', 'num_sats', 'mths_since_rcnt_il', 'collections_12_mths_ex_med', 'mths_since_recent_inq', 'int_rate', 'num_bc_sats', 'total_il_high_credit_limit', 'verification_status', 'num_actv_bc